In [419]:
# importamos las librerías que necesitamos

# Tratamiento de datos
# -----------------------------------------------------------------------
import pandas as pd
import numpy as np
from src import soporte_datasets as sp_datasets

# Visualización
# -----------------------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns

# Imputación de nulos usando métodos avanzados estadísticos
# -----------------------------------------------------------------------
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.impute import KNNImputer


# Configuración
# -----------------------------------------------------------------------
pd.set_option('display.max_columns', None) # para poder visualizar todas las columnas de los DataFrames

# Gestión de los warnings
# -----------------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore")

In [420]:
df_flight = pd.read_csv('files/Customer Flight Activity.csv') 
df_flight.head(2)

,Loyalty Number,Year,Month,Flights Booked,Flights with Companions,Total Flights,Distance,Points Accumulated,Points Redeemed,Dollar Cost Points Redeemed
0,100018,2017,1,3,0,3,1521,152.0,0,0
1,100102,2017,1,10,4,14,2030,203.0,0,0


In [421]:
df_loyalty = pd.read_csv('files/Customer Loyalty History.csv') #,index_col=0
df_loyalty.head(2)

,Loyalty Number,Country,Province,City,Postal Code,Gender,Education,Salary,Marital Status,Loyalty Card,CLV,Enrollment Type,Enrollment Year,Enrollment Month,Cancellation Year,Cancellation Month
0,480934,Canada,Ontario,Toronto,M2Z 4K1,Female,Bachelor,83236.0,Married,Star,3839.14,Standard,2016,2,NaN,NaN
1,549612,Canada,Alberta,Edmonton,T3G 6Y6,Male,College,NaN,Divorced,Star,3839.61,Standard,2016,3,NaN,NaN


Lo primero a realizar es poner todos los datos en minúsculas, y en las columnas, cambiar los espacios por guiones para un mejor tratamiento de los datos. 

In [422]:
df_flight.columns = sp_datasets.min_datos(df_flight)

In [423]:
df_flight.head()

,loyalty_number,year,month,flights_booked,flights_with_companions,total_flights,distance,points_accumulated,points_redeemed,dollar_cost_points_redeemed
0,100018,2017,1,3,0,3,1521,152.0,0,0
1,100102,2017,1,10,4,14,2030,203.0,0,0
2,100140,2017,1,6,0,6,1200,120.0,0,0
3,100214,2017,1,0,0,0,0,0.0,0,0
4,100272,2017,1,0,0,0,0,0.0,0,0


In [424]:
df_loyalty.columns = sp_datasets.min_datos(df_loyalty)

In [425]:
df_loyalty.head()

,loyalty_number,country,province,city,postal_code,gender,education,salary,marital_status,loyalty_card,clv,enrollment_type,enrollment_year,enrollment_month,cancellation_year,cancellation_month
0,480934,Canada,Ontario,Toronto,M2Z 4K1,Female,Bachelor,83236.0,Married,Star,3839.14,Standard,2016,2,NaN,NaN
1,549612,Canada,Alberta,Edmonton,T3G 6Y6,Male,College,NaN,Divorced,Star,3839.61,Standard,2016,3,NaN,NaN
2,429460,Canada,British Columbia,Vancouver,V6E 3D9,Male,College,NaN,Single,Star,3839.75,Standard,2014,7,2018.0,1.0
3,608370,Canada,Ontario,Toronto,P1W 1K4,Male,College,NaN,Single,Star,3839.75,Standard,2013,2,NaN,NaN
4,530508,Canada,Quebec,Hull,J8Y 3Z5,Male,Bachelor,103495.0,Married,Star,3842.79,Standard,2014,10,NaN,NaN


# Gestión de valores nulos

Tal como hemos indicado en la fase inicial, existen nulos en las columnas de Salario y cancelación, vamos a tratarlos antes de la unión.

    - Para las columnas de cancelación de df_loyalty, vamos a pasar los Nulos como False, manteniendo el año y mes, ya que los valores nulos significan que el usuario sigue estando dado de alta en el sistema.

    - Para la columna Salario de df_loyalty, analizaremos patrones para ver la mejor opción.

In [426]:
df_loyalty["cancellation_year"]= sp_datasets.nulos_false_int(df_loyalty["cancellation_year"])
df_loyalty["cancellation_month"]= sp_datasets.nulos_false_int(df_loyalty["cancellation_month"])

In [427]:
df_loyalty.head()

,loyalty_number,country,province,city,postal_code,gender,education,salary,marital_status,loyalty_card,clv,enrollment_type,enrollment_year,enrollment_month,cancellation_year,cancellation_month
0,480934,Canada,Ontario,Toronto,M2Z 4K1,Female,Bachelor,83236.0,Married,Star,3839.14,Standard,2016,2,False,False
1,549612,Canada,Alberta,Edmonton,T3G 6Y6,Male,College,NaN,Divorced,Star,3839.61,Standard,2016,3,False,False
2,429460,Canada,British Columbia,Vancouver,V6E 3D9,Male,College,NaN,Single,Star,3839.75,Standard,2014,7,2018,1
3,608370,Canada,Ontario,Toronto,P1W 1K4,Male,College,NaN,Single,Star,3839.75,Standard,2013,2,False,False
4,530508,Canada,Quebec,Hull,J8Y 3Z5,Male,Bachelor,103495.0,Married,Star,3842.79,Standard,2014,10,False,False


In [428]:
(df_loyalty.isna().sum()/df_loyalty.shape[0]*100).sort_values(ascending=False).head(2)

#Comprobamos que solo está la columna Salario.

salary            25.321145
loyalty_number     0.000000
dtype: float64

In [429]:
df_loyalty.info()

#comprobamos que las columnas que acabamos de procesar es tipo objeto, tienen valores numericos y 
# boleanos, de momento vamos a dejarlo así por si tenemos que analizar % por mes.


<class 'pandas.DataFrame'>
RangeIndex: 16737 entries, 0 to 16736
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   loyalty_number      16737 non-null  int64  
 1   country             16737 non-null  str    
 2   province            16737 non-null  str    
 3   city                16737 non-null  str    
 4   postal_code         16737 non-null  str    
 5   gender              16737 non-null  str    
 6   education           16737 non-null  str    
 7   salary              12499 non-null  float64
 8   marital_status      16737 non-null  str    
 9   loyalty_card        16737 non-null  str    
 10  clv                 16737 non-null  float64
 11  enrollment_type     16737 non-null  str    
 12  enrollment_year     16737 non-null  int64  
 13  enrollment_month    16737 non-null  int64  
 14  cancellation_year   16737 non-null  object 
 15  cancellation_month  16737 non-null  object 
dtypes: float64(2), 

Para el salario, vamos a ver los salarios nulos con la Educación para ver si tienen patrón.

In [430]:
salario_negativo = df_loyalty[df_loyalty["salary"]<0]

In [431]:
salario_negativo["salary"].unique()
#Analizamos los salarios negativos.

array([-49830., -12497., -46683., -45962., -19325., -43234., -10605.,
       -17534., -58486., -31911., -49001., -34079.,  -9081., -46470.,
       -26322., -47310., -39503., -19332., -46303., -57297.])

In [432]:
len(salario_negativo["salary"].unique())
#Analizamos cuántos son

20

In [433]:
salario_negativo['education'].value_counts()

#los datos corresponden a la mayoría de educación Bachelor.

education
Bachelor                19
High School or Below     1
Name: count, dtype: int64

In [434]:
salario_negativo[["education","salary"]].sort_values("salary")

,education,salary
7373,Bachelor,-58486.0
16735,Bachelor,-57297.0
1082,High School or Below,-49830.0
8767,Bachelor,-49001.0
14327,Bachelor,-47310.0
2471,Bachelor,-46683.0
12596,Bachelor,-46470.0
16431,Bachelor,-46303.0
3575,Bachelor,-45962.0
4712,Bachelor,-43234.0


Ahora que tengo los salarios negativos, voy a verlos junto con la educación

In [435]:
media_salarios_limpios = df_loyalty[df_loyalty['salary'] > 0].groupby('education')['salary'].min().reset_index()
media_salarios_limpios.head()
#saco el valor mínimo de los salarios por nivel educativo.

,education,salary
0,Bachelor,15609.0
1,Doctor,48109.0
2,High School or Below,21853.0
3,Master,56414.0


Apreciamos que el valor minimo de Bachelor es 15609, y el valor negativo más bajo es 9081, seguido de 10605, interpretamos que la imputación de los datos ha habido una errata en el signo y los volvemos positivos, teniendo en cuenta que el valor de 9081 sea un posible outlier.

In [436]:
# Aplicamos el valor absoluto a la columna salary
df_loyalty['salary'] = df_loyalty['salary'].abs()

# Verificamos que ya no hay negativos
df_loyalty['salary'].min()

np.float64(9081.0)

In [437]:
df_loyalty.describe(include=np.number).T

,count,mean,std,min,25%,50%,75%,max
loyalty_number,16737.0,549735.880445,258912.132453,100018.00,326603.00,550434.00,772019.00,999986.00
salary,12499.0,79359.340907,34749.691464,9081.00,59246.50,73455.00,88517.50,407228.00
clv,16737.0,7988.896536,6860.982280,1898.01,3980.84,5780.18,8940.58,83325.38
enrollment_year,16737.0,2015.253211,1.979111,2012.00,2014.00,2015.00,2017.00,2018.00
enrollment_month,16737.0,6.669116,3.398958,1.00,4.00,7.00,10.00,12.00


Tras poner los valores positivos, vemos que la columna `Salary` tiene un rango muy amplio, desde los 9081 hasta los 407228, la mediana y media están próximas y la desviación estándar es muy alta, lo que podría interpertarse como dispersión en los datos.

In [438]:
cv = df_loyalty["salary"].std()/df_loyalty["salary"].mean()

cv

np.float64(0.43787777300716524)

Como tengo valores nulos en Salario, voy a realizar el coeficiente de variación de la columna para luego ver si sustituyendo los datos, se modifica mucho.

El cv es 0.43 (~ 43%), lo que indica que los datos son heterogéneos, hay mucha dispersión.

In [439]:
#Hago la media de salario por tipo de educación y vemos que el "College" no tiene valores.
round(df_loyalty.groupby("education")["salary"].mean(),2)

#orden de menor a mayor: high school < college < bachelor < master < doctor

education
Bachelor                 72577.25
College                       NaN
Doctor                  178608.90
High School or Below     61199.16
Master                  103757.85
Name: salary, dtype: float64

In [440]:
round(df_loyalty.groupby("education")["salary"].median(),2)

#College tiene todos los datos nulos en su nivel educativo, no podemos usarlo para crear un patron.

education
Bachelor                 71960.0
College                      NaN
Doctor                  182143.5
High School or Below     61915.0
Master                  105487.0
Name: salary, dtype: float64

Vamos a usar el CV inicial para ver cuál es la mejor opción de sustitución:

In [441]:
# Calculamos la mediana y media global de salario
mediana_global = df_loyalty["salary"].median()
media_global = df_loyalty["salary"].mean()

# Creamos dos columnas, uno imputando la media global de salario, otra la mediana global 

df_loyalty["salary_media"] = df_loyalty["salary"].fillna(media_global)
df_loyalty["salary_mediana"] = df_loyalty["salary"].fillna(mediana_global)

# Recalculamos los estadísticos
mean_imputado = df_loyalty["salary_media"].mean()
std_imputado = df_loyalty["salary_media"].std()
cv_imputado = std_imputado / mean_imputado

mean_imputado2 = df_loyalty["salary_mediana"].mean()
std_imputado2 = df_loyalty["salary_mediana"].std()
cv_imputado2 = std_imputado2 / mean_imputado2


print(f"CV Real: {cv:.2f}")
print(f"CV tras Imputar media global de Salary: {cv_imputado:.2f}")
print(f"CV tras Imputar mediana global de Salary: {cv_imputado2:.2f}")


CV Real: 0.44
CV tras Imputar media global de Salary: 0.38
CV tras Imputar mediana global de Salary: 0.39


Creemos que tanto la media como la mediana global de salary va a sesgar los perfiles de los clientes, como la educación tiene una jerarquía, vamos a sacar un punto medio entre los dos niveles que se encuentra "College"

In [442]:
#creamos un df con la media de los salarios por nivel educativo.
medianas_educacion = df_loyalty.groupby("education")["salary"].median()

# high school < college < bachelor
mediana_hschool = medianas_educacion['High School or Below']
mediana_bach = medianas_educacion['Bachelor']
sueldo_college_estimado = (mediana_hschool + mediana_bach) / 2

sueldo_college_estimado

np.float64(66937.5)

In [443]:
df_loyalty["salary_estimado"] = df_loyalty["salary"].fillna(sueldo_college_estimado)

mean_imputado3 = df_loyalty["salary_estimado"].mean()
std_imputado3= df_loyalty["salary_estimado"].std()
cv_imputado3 = std_imputado3 / mean_imputado3


print(f"CV Real: {cv:.2f}")
print(f"CV tras Imputar media global de Salary: {cv_imputado:.2f}")
print(f"CV tras Imputar mediana global de Salary: {cv_imputado2:.2f}")
print(f"CV tras Imputar mediana promedio por nivel educativo/Salary: {cv_imputado3:.2f}")

CV Real: 0.44
CV tras Imputar media global de Salary: 0.38
CV tras Imputar mediana global de Salary: 0.39
CV tras Imputar mediana promedio por nivel educativo/Salary: 0.40


el coeficiente de variación del salario promedio entre salario y educación de los niveles anterior y superior de "College" se acerca más a los datos, por lo tanto, nos quedaremos con esa opción.

In [444]:
df_loyalty.head()

,loyalty_number,country,province,city,postal_code,gender,education,salary,marital_status,loyalty_card,clv,enrollment_type,enrollment_year,enrollment_month,cancellation_year,cancellation_month,salary_media,salary_mediana,salary_estimado
0,480934,Canada,Ontario,Toronto,M2Z 4K1,Female,Bachelor,83236.0,Married,Star,3839.14,Standard,2016,2,False,False,83236.000000,83236.0,83236.0
1,549612,Canada,Alberta,Edmonton,T3G 6Y6,Male,College,NaN,Divorced,Star,3839.61,Standard,2016,3,False,False,79359.340907,73455.0,66937.5
2,429460,Canada,British Columbia,Vancouver,V6E 3D9,Male,College,NaN,Single,Star,3839.75,Standard,2014,7,2018,1,79359.340907,73455.0,66937.5
3,608370,Canada,Ontario,Toronto,P1W 1K4,Male,College,NaN,Single,Star,3839.75,Standard,2013,2,False,False,79359.340907,73455.0,66937.5
4,530508,Canada,Quebec,Hull,J8Y 3Z5,Male,Bachelor,103495.0,Married,Star,3842.79,Standard,2014,10,False,False,103495.000000,103495.0,103495.0


In [445]:
df_loyalty.describe(include=np.number).T #para variables numéricas

,count,mean,std,min,25%,50%,75%,max
loyalty_number,16737.0,549735.880445,258912.132453,100018.00,326603.00,550434.000000,772019.00,999986.00
salary,12499.0,79359.340907,34749.691464,9081.00,59246.50,73455.000000,88517.50,407228.00
clv,16737.0,7988.896536,6860.982280,1898.01,3980.84,5780.180000,8940.58,83325.38
enrollment_year,16737.0,2015.253211,1.979111,2012.00,2014.00,2015.000000,2017.00,2018.00
enrollment_month,16737.0,6.669116,3.398958,1.00,4.00,7.000000,10.00,12.00
salary_media,16737.0,79359.340907,30029.311812,9081.00,63899.00,79359.340907,82940.00,407228.00
salary_mediana,16737.0,77864.294198,30138.879584,9081.00,63899.00,73455.000000,82940.00,407228.00
salary_estimado,16737.0,76213.988588,30511.295223,9081.00,63899.00,66937.500000,82940.00,407228.00


Antes de borrar columnas, vemos lo siguiente:

    - Si lo cambiamos por la media global, el 50% que antes cobraba 73k ahora pasa a 79k, la mediana se ha desplazado mucho.
    - Entre la mediana global y el promedio de nivel educativo, hay una pequeña diferencia en la desviación estándar, media y mediana, pero vamos a optar por dejar el salario estimado para mantener una coherencia de datos, donde high school < college < bachelor.

In [446]:
# Eliminamos todas las columnas provisionales de salary creadas.
df_loyalty.drop(columns=['salary', 'salary_mediana', 'salary_media'], inplace=True, errors='ignore')


In [447]:
df_loyalty.head()

,loyalty_number,country,province,city,postal_code,gender,education,marital_status,loyalty_card,clv,enrollment_type,enrollment_year,enrollment_month,cancellation_year,cancellation_month,salary_estimado
0,480934,Canada,Ontario,Toronto,M2Z 4K1,Female,Bachelor,Married,Star,3839.14,Standard,2016,2,False,False,83236.0
1,549612,Canada,Alberta,Edmonton,T3G 6Y6,Male,College,Divorced,Star,3839.61,Standard,2016,3,False,False,66937.5
2,429460,Canada,British Columbia,Vancouver,V6E 3D9,Male,College,Single,Star,3839.75,Standard,2014,7,2018,1,66937.5
3,608370,Canada,Ontario,Toronto,P1W 1K4,Male,College,Single,Star,3839.75,Standard,2013,2,False,False,66937.5
4,530508,Canada,Quebec,Hull,J8Y 3Z5,Male,Bachelor,Married,Star,3842.79,Standard,2014,10,False,False,103495.0


In [451]:
df_loyalty.rename(columns={"salary_estimado": "salary"}, inplace=True)

In [452]:
df_loyalty.head()

,loyalty_number,country,province,city,postal_code,gender,education,marital_status,loyalty_card,clv,enrollment_type,enrollment_year,enrollment_month,cancellation_year,cancellation_month,salary
0,480934,Canada,Ontario,Toronto,M2Z 4K1,Female,Bachelor,Married,Star,3839.14,Standard,2016,2,False,False,83236.0
1,549612,Canada,Alberta,Edmonton,T3G 6Y6,Male,College,Divorced,Star,3839.61,Standard,2016,3,False,False,66937.5
2,429460,Canada,British Columbia,Vancouver,V6E 3D9,Male,College,Single,Star,3839.75,Standard,2014,7,2018,1,66937.5
3,608370,Canada,Ontario,Toronto,P1W 1K4,Male,College,Single,Star,3839.75,Standard,2013,2,False,False,66937.5
4,530508,Canada,Quebec,Hull,J8Y 3Z5,Male,Bachelor,Married,Star,3842.79,Standard,2014,10,False,False,103495.0


In [455]:
print(f"Comprobamos nulos: {df_loyalty.isna().sum()}")

Comprobamos nulos: loyalty_number        0
country               0
province              0
city                  0
postal_code           0
gender                0
education             0
marital_status        0
loyalty_card          0
clv                   0
enrollment_type       0
enrollment_year       0
enrollment_month      0
cancellation_year     0
cancellation_month    0
salary                0
dtype: int64


Vamos ahora a tratar los valores duplicados del `df_flight`